# MANTA - Embedding Tissue Sections

MANTA's non-rigid registration procedure requires the tissue sections that are being aligned to be embedded in a latent space that combines spatial and molecular features. This notebook embeds to tissue slices in preparation for a non-rigid alignment.

In [ ]:
import anndata as ad
import manta

# I/O
from pathlib import Path

In [ ]:
%load_ext autoreload
%autoreload 2

## Preprocessing

See `preprocessing.ipynb` for details.

In [ ]:
data_dir = Path("/staging/leuven/stg_00002/lcb/lcb_projects/SPF/mkovacic/data/dataset_MOUSE_BRAIN")

In [ ]:
source = ad.read_h5ad(data_dir / "slice_75.h5ad")
target = ad.read_h5ad(data_dir / "slice_76.h5ad")

In [ ]:
adatas = manta.preprocessing.preprocess(
    source=source,
    target=target,
    gene_key="gene_name",
    spatial_key="X_spatial_coords",
    max_harmony_iterations=25
)

In [ ]:
source = adatas[0]
target = adatas[1]

## Rigid Alignment

See `rigid_alignment.ipynb` for details.

In [ ]:
manta.alignment.rigid(
    source,
    target,
    voxel_scales=[2500, 1000, 500, 250, 125],
    spatial_key="spatial_manta",
    expression_key="X_pca"
)

## Embedding

Embedding the tissues consists of two distinct steps:

1. Subsampling the tissue sections.
2. Embedding the subsampled observations and their spatial/molecular information in a shared latent space across **all** input sections.

### Sampling 

Sampling is already covered in `preprocessing.ipynb` and **is required** for embedding. MANTA will embed only the subsampled tissue to save memory and compute.

**NOTE**: *Make sure to sample from the correct `anndata.obsm` channel, in this case `rigid` since we already computed a rigid alignment first!*

In [ ]:
manta.sampling.sample(
    adatas=adatas,
    sampler="importance",
    fraction=0.25,
    bin_size=128,
    spatial_key="rigid"
)

In [ ]:
manta.plot.sampling(
    adatas=adatas,
    labels=["source", "target"],
    colors=["red", "blue"],
    sampling_key="sampling",

    figsize=(6, 6),
    alpha=0.45,
    marker_size=6,
    equal_aspect=True
)

### Embedding

An embedding can be obtained by through `manta.embedding.embed`.

**NOTE**: *Multiple tissue sections can be embedded simultaneously. This is especially helpful when aligning many tissue sections s.t. the embedding step only has to be performed once.*

In [ ]:
manta.embedding.embed(
    adatas=adatas,

    spatial_key="rigid",
    sampling_key="sampling",

    pca_basis_key="X_pca",
    nmf_basis_key="X_nmf"
)